In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from xgboost import XGBClassifier

from variables import ajouter_contexte
from evaluation import scores_classification, seuil_f1

h = pd.read_parquet("../data/processed/arcep_2022_2025.parquet")
web = h.query("protocole == 'WEB'").copy()
web["echec"] = (1 - web["succes"]).astype(int)
web = ajouter_contexte(web)

y = web["echec"]
annee = web["annee"]
entr, val, test = annee <= 2023, annee == 2024, annee == 2025

In [2]:
CATEGORIES = ["operator", "zone", "situation"]
JEUX = {
    "base":       (["rsrp", "heure"], ["rsrp_absent"]),
    "prédiction": (["rsrp", "heure", "echec_prec"], ["rsrp_absent", "prec_absent"]),
    "diagnostic": (["rsrp", "heure", "echec_prec", "taux_echec_autres"], ["rsrp_absent", "prec_absent"]),
}

def fabriquer(nom_modele, numeriques, booleennes):
    if nom_modele == "logistique":
        prep = ColumnTransformer([
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                              ("echelle", StandardScaler())]), numeriques),
            ("bool", "passthrough", booleennes),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIES)])
        clf = LogisticRegression(max_iter=1000)
    else:
        prep = ColumnTransformer([
            ("num", "passthrough", numeriques + booleennes),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORIES)])
        clf = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                            n_jobs=-1, random_state=0)
    return Pipeline([("prep", prep), ("clf", clf)])

resultats = {}
for jeu, (numeriques, booleennes) in JEUX.items():
    X = web[numeriques + booleennes + CATEGORIES]
    for nom_modele in ["logistique", "xgboost"]:
        modele = fabriquer(nom_modele, numeriques, booleennes)
        modele.fit(X[entr], y[entr])                                   # 1. entraîner sur 2022-2023
        seuil = seuil_f1(y[val], modele.predict_proba(X[val])[:, 1])   # 2. régler le seuil sur 2024
        final = clone(modele).fit(X[entr | val], y[entr | val])        # 3. réentraîner sur 2022-2024
        p = final.predict_proba(X[test])[:, 1]                         # 4. tester sur 2025
        resultats[(jeu, nom_modele)] = scores_classification(y[test], p, seuil) | {"seuil": round(seuil, 2)}
        print(jeu, nom_modele, "terminé")

regle = ((web["rsrp"] < -120) | web["rsrp_absent"]).astype(int)
resultats[("règle", "RSRP")] = scores_classification(y[test], regle[test])
pd.DataFrame(resultats).T

base logistique terminé
base xgboost terminé
prédiction logistique terminé
prédiction xgboost terminé
diagnostic logistique terminé
diagnostic xgboost terminé


precision  rappel     f1     ap  seuil
base       logistique      0.793   0.564  0.659  0.720   0.41
           xgboost         0.775   0.593  0.672  0.700   0.46
prédiction logistique      0.789   0.676  0.728  0.771   0.39
           xgboost         0.786   0.688  0.734  0.790   0.48
diagnostic logistique      0.769   0.712  0.740  0.796   0.35
           xgboost         0.787   0.688  0.734  0.803   0.45
règle      RSRP            0.724   0.645  0.682  0.499    NaN

In [3]:
from sklearn.inspection import permutation_importance

numeriques, booleennes = JEUX["prédiction"]
X = web[numeriques + booleennes + CATEGORIES]
modele = fabriquer("xgboost", numeriques, booleennes)
modele.fit(X[annee <= 2024], y[annee <= 2024])

imp = permutation_importance(modele, X[test], y[test], scoring="average_precision",
                             n_repeats=3, random_state=0)
importance = pd.DataFrame({"baisse_AP": imp.importances_mean, "ecart_type": imp.importances_std},
                          index=X.columns).sort_values("baisse_AP", ascending=False).round(4)
importance

,baisse_AP,ecart_type
rsrp,0.4554,0.0007
echec_prec,0.3159,0.0014
zone,0.0101,0.0004
operator,0.0083,0.0005
situation,0.0018,0.0002
rsrp_absent,0.0000,0.0000
prec_absent,0.0000,0.0000
heure,-0.0003,0.0004


In [4]:
pd.DataFrame(resultats).T.to_csv("../results/tables/web_variables_contexte_test_2025.csv")
importance.to_csv("../results/tables/web_importance_permutation.csv")